# Analyse A/B Testing contribution

A/B Testing pour valider la meilleur façon pour que l'utilisateur accès à un contenu. 

On a 3 variants : 

 * original : version actuelle du site
 * regular_button : on propose par défaut de rechercher son entreprise
 * radio_button : on ajoute l'option où l'utilisateur peut ne pas renseigner sa CC

In [ ]:
%load_ext dotenv
%dotenv

In [ ]:
import pandas as pd
from analysis.connectors.matomo import MatomoSQLConnector

matomo = MatomoSQLConnector()
await matomo.connect()

In [ ]:
# - 10 juin / now
interval_start = '2026-06-12 00:00:00'
interval_stop = '2026-07-04 00:00:00'

In [ ]:
columns = ['action_id',
    'idvisit',
    'actions',
    'operatingsystemname',
    'action_type',
    'action_eventcategory',
    'action_eventaction',
    'action_eventname',
    'action_eventvalue',
    'action_url',
    'experiments']

range_query = f"""
        SELECT {", ".join(columns)} FROM matomo_partitioned
        WHERE action_timestamp >= '{interval_start}'
          AND action_timestamp < '{interval_stop}'
        """

In [ ]:
query_visits =  range_query + f"""
          ORDER BY action_timestamp asc;
    """

visits_data = await matomo.run_query(query_visits)

In [ ]:
visits_df = pd.DataFrame(visits_data, columns=columns)

In [ ]:
base_slugs = [
    "quel-est-le-salaire-minimum",
    "quel-est-le-salaire-minimum-dun-alternant-en-2026",
    "le-preavis-de-licenciement-doit-il-etre-execute-en-totalite-y-compris-si-le-salarie-a-retrouve-un-emploi",
    "quelle-est-la-duree-du-conge-de-maternite",
    "dans-le-cadre-dun-cdd-quel-est-le-montant-de-lindemnite-de-fin-de-contrat",
    "quelle-est-la-duree-maximale-du-contrat-de-mission-interim",
    "quelle-est-la-duree-de-preavis-en-cas-de-depart-a-la-retraite",
    "si-le-salarie-est-malade-pendant-ses-conges-quelles-en-sont-les-consequences",
    "si-un-poste-se-libere-ou-est-cree-dans-lentreprise-lemployeur-doit-il-en-informer-les-salaries-ou-le-leur-proposer-en-priorite",
    "heures-supplementaires",
    "quelles-sont-les-conditions-de-cumul-demplois",
    "conges-supplementaires-pour-anciennete",
    "faut-il-respecter-un-delai-de-carence-entre-deux-cdd-si-oui-quelle-est-sa-duree",
    "faut-il-respecter-un-delai-de-carence-entre-deux-contrats-de-mission-interim",
    "le-salarie-peut-il-sabsenter-pour-rechercher-un-emploi-pendant-son-preavis",
    "les-conges-pour-evenements-familiaux",
    "lentreprise-peut-elle-embaucher-dans-le-cadre-dun-cdi-de-chantier-ou-doperation",
    "quelle-peut-etre-la-duree-maximale-dun-cdd",
    "comment-determiner-lanciennete-du-salarie",
    "a-quelles-indemnites-peut-pretendre-un-salarie-qui-part-a-la-retraite",
    "embauche-en-contrat-dextra-cdd-dusage",
    "quelles-informations-doivent-figurer-dans-le-contrat-de-travail-ou-la-lettre-dengagement",
    "quelles-sont-les-conditions-dattribution-de-la-prime-pour-travaux-dangereux-et-de-la-prime-pour-travaux-insalubres",
    "en-cas-de-perte-de-marche-par-lemployeur-quelles-sont-les-conditions-dun-transfert-des-contrats-de-travail",
    "arret-maladie-pendant-la-periode-dessai-quelles-sont-les-regles",
    "quest-ce-quune-rupture-conventionnelle",
    "combien-de-fois-le-contrat-de-travail-peut-il-etre-renouvele",
    "arret-maladie-pendant-le-preavis-quelles-consequences",
    "quelles-sont-les-primes-prevues-par-la-convention-collective",
    "quand-le-salarie-a-t-il-droit-a-une-prime-danciennete-quel-est-son-montant",
    "dans-le-cadre-dun-contrat-de-mission-interim-quel-est-le-montant-de-lindemnite-de-fin-de-contrat",
    "quelles-sont-les-consequences-du-non-respect-du-preavis-par-le-salarie-ou-lemployeur",
    "quelle-est-la-duree-maximale-de-la-periode-dessai-sans-et-avec-renouvellement",
    "la-periode-dessai-peut-elle-etre-renouvelee",
    "quelles-sont-les-conditions-de-la-clause-de-non-concurrence",
    "quelle-est-la-duree-de-preavis-en-cas-de-mise-a-la-retraite",
    "travail-du-dimanche-quelle-contrepartie",
    "quelle-est-la-duree-du-preavis-en-cas-de-demission",
    "en-cas-de-maladie-le-salarie-a-t-il-droit-a-une-garantie-demploi",
    "quelles-sont-les-consequences-du-deces-de-lemployeur-sur-le-contrat-de-travail",
    "jours-feries-et-ponts-dans-le-secteur-prive",
    "quelle-est-la-duree-de-preavis-en-cas-de-licenciement",
    "est-il-obligatoire-davoir-un-contrat-de-travail-ecrit-et-signe",
    "le-preavis-de-demission-doit-il-etre-execute-en-totalite-y-compris-si-le-salarie-a-retrouve-un-emploi",
    "en-cas-darret-maladie-du-salarie-lemployeur-doit-il-assurer-le-maintien-de-salaire",
    "quelles-sont-les-conditions-dindemnisation-pendant-le-conge-de-maternite",
]

## Récupération du taux de completion pour chaque variant

Pour le nombre de page vue, on veut regarder les utilisateurs ayant affiché le contenu (cc ou cdt)
Exp : [{"name": "LabelCardSearch", "variation": {"name": "fiche-pratique", "idvariation": 7}, "idexperiment": "5"}, {"name": "contribution_afficher_info", "variation": {"name": "Version originale", "idvariation": 0}, "idexperiment": "8"}]

## Extraction du variant contribution_afficher_info

In [ ]:
import json

EXPERIMENT_NAME = "contribution_afficher_info"

def extract_variant(experiments):
    if experiments is None:
        return None
    if isinstance(experiments, str):
        try:
            experiments = json.loads(experiments)
        except (json.JSONDecodeError, TypeError):
            return None
    if not isinstance(experiments, list):
        return None
    for exp in experiments:
        if isinstance(exp, dict) and exp.get("name") == EXPERIMENT_NAME:
            return exp.get("variation", {}).get("name")
    return None

visits_df["variant"] = visits_df["experiments"].apply(extract_variant)

# Propagation du variant au niveau de la visite
# (première valeur non nulle trouvée pour chaque idvisit)
variant_by_visit = (
    visits_df.dropna(subset=["variant"])
    .groupby("idvisit")["variant"]
    .first()
)
visits_df["variant"] = visits_df["idvisit"].map(variant_by_visit)

## Extraction du slug depuis action_url

In [ ]:
import re

URL_PATTERN = re.compile(
    r"^https://code\.travail\.gouv\.fr/contribution/([^/?#]+)(?:[?#].*)?$"
)

def extract_slug(url):
    if not isinstance(url, str):
        return None
    m = URL_PATTERN.match(url)
    return m.group(1) if m else None

visits_df["slug"] = visits_df["action_url"].apply(extract_slug)

## Agrégation : visites uniques par slug × variant

In [ ]:
pages_df = visits_df[
    (visits_df["action_type"] == "action")
    & (visits_df["slug"].isin(base_slugs))
].copy()

# Les visites sans variant connu sont regroupées dans une colonne dédiée
pages_df["variant"] = pages_df["variant"].fillna("(sans variant)")

result = (
    pages_df.groupby(["slug", "variant"])["idvisit"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(base_slugs, fill_value=0)  # garde l'ordre de ta liste, slugs absents à 0
)

result

## Extraction du slug depuis le nom d'event

In [ ]:
EVENTNAME_PATTERN = re.compile(
    r"^/contribution/([^/?#|]+)(?:\|.*)?$"
)

def extract_slug_from_eventname(name):
    if not isinstance(name, str):
        return None
    m = EVENTNAME_PATTERN.match(name)
    return m.group(1) if m else None

## Fonction générique de comptage de visites uniques pour un event

Comme on a deux events à traiter (et probablement d'autres à venir), autant factoriser. On réutilise la colonne variant déjà propagée par idvisit à l'étape précédente :

In [ ]:
def unique_visits_for_event(df, category, action):
    mask = (
        (df["action_type"] == "event")
        & (df["action_eventcategory"] == category)
        & (df["action_eventaction"] == action)
    )
    events = df[mask].copy()
    events["slug"] = events["action_eventname"].apply(extract_slug_from_eventname)
    events = events[events["slug"].isin(base_slugs)]
    events["variant"] = events["variant"].fillna("(sans variant)")

    return (
        events.groupby(["slug", "variant"])["idvisit"]
        .nunique()  # une visite = 1, même si l'event est déclenché plusieurs fois
        .unstack(fill_value=0)
        .reindex(base_slugs, fill_value=0)
    )

## Calcul des deux métriques et assemblage avec le tableau de visites

In [ ]:
consult_cdt = unique_visits_for_event(
    visits_df, "cc_search_type_of_users", "click_p3"
)
consult_cc = unique_visits_for_event(
    visits_df, "contribution", "click_afficher_les_informations_CC"
)

result_full = pd.concat(
    {
        "visites": result,          # tableau de l'étape précédente
        "consult_cdt": consult_cdt,
        "consult_cc": consult_cc,
    },
    axis=1,
).fillna(0).astype(int)

# Optionnel : regrouper les colonnes par variant plutôt que par métrique
result_full = result_full.swaplevel(axis=1).sort_index(axis=1)

result_full

## Refactorisation : on isole les dataframes filtrés (slug, variant, idvisit)

In [ ]:
def filter_event_df(df, category, action):
    mask = (
        (df["action_type"] == "event")
        & (df["action_eventcategory"] == category)
        & (df["action_eventaction"] == action)
    )
    events = df[mask].copy()
    events["slug"] = events["action_eventname"].apply(extract_slug_from_eventname)
    events = events[events["slug"].isin(base_slugs)]
    events["variant"] = events["variant"].fillna("(sans variant)")
    return events[["slug", "variant", "idvisit"]]

# Page views (réutilise le pages_df de la 1re étape, ou le reconstruit)
pages = pages_df[["slug", "variant", "idvisit"]]

cdt_events = filter_event_df(visits_df, "cc_search_type_of_users", "click_p3")
cc_events = filter_event_df(visits_df, "contribution", "click_afficher_les_informations_CC")

# Union CDT ou CC : le concat + nunique déduplique automatiquement les idvisit
any_events = pd.concat([cdt_events, cc_events])

## Table de comptage avec ligne TOTAL

Point important pour la ligne TOTAL : je la recalcule avec un nunique global par variant, plutôt que de sommer les lignes. Une visite qui a consulté 3 slugs compte 3 fois dans la somme des lignes, mais 1 fois dans le total dédupliqué. Le TOTAL peut donc être inférieur à la somme de la colonne — c'est voulu.

In [ ]:
def build_count_table(df):
    counts = (
        df.groupby(["slug", "variant"])["idvisit"]
        .nunique()
        .unstack(fill_value=0)
        .reindex(base_slugs, fill_value=0)
    )
    counts.loc["TOTAL"] = df.groupby("variant")["idvisit"].nunique()
    return counts.fillna(0).astype(int)

visites_t = build_count_table(pages)
cdt_t = build_count_table(cdt_events)
cc_t = build_count_table(cc_events)
any_t = build_count_table(any_events)

In [ ]:
def safe_rate(num, den):
    return (num / den.replace(0, pd.NA) * 100).round(1).fillna(0)

result_full = pd.concat(
    {
        "visites": visites_t,
        "consult_cdt": cdt_t,
        "taux_cdt_%": safe_rate(cdt_t, visites_t),
        "consult_cc": cc_t,
        "taux_cc_%": safe_rate(cc_t, visites_t),
        "taux_global_%": safe_rate(any_t, visites_t),
    },
    axis=1,
)

# Colonnes regroupées par variant : (variant, métrique)
result_full = result_full.swaplevel(axis=1).sort_index(axis=1, level=0, sort_remaining=False)

result_full

In [ ]:
def print_totaux_par_variant(visites_t, cdt_t, cc_t, any_t):
    tot_visites = visites_t.loc["TOTAL"]
    tot_cdt = cdt_t.loc["TOTAL"]
    tot_cc = cc_t.loc["TOTAL"]
    tot_any = any_t.loc["TOTAL"]

    # Union des variants présents dans les 4 tables
    variants = sorted(
        set(tot_visites.index)
        | set(tot_cdt.index)
        | set(tot_cc.index)
        | set(tot_any.index)
    )

    for variant in variants:
        visites = int(tot_visites.get(variant, 0))
        cdt = int(tot_cdt.get(variant, 0))
        cc = int(tot_cc.get(variant, 0))
        any_ = int(tot_any.get(variant, 0))

        taux_cdt = cdt / visites * 100 if visites else 0
        taux_cc = cc / visites * 100 if visites else 0
        taux_global = any_ / visites * 100 if visites else 0

        print(f"=== {variant} ===")
        print(f"  Visites uniques      : {visites}")
        print(f"  Consult. CDT         : {cdt} ({taux_cdt:.1f} %)")
        print(f"  Consult. CC          : {cc} ({taux_cc:.1f} %)")
        print(f"  Consult. CDT ou CC   : {any_} (taux global : {taux_global:.1f} %)")
        print()

print_totaux_par_variant(visites_t, cdt_t, cc_t, any_t)

In [ ]:
def print_detail_slug(slug, visites_t, cdt_t, cc_t, any_t):
    variants = sorted(
        (
            set(visites_t.columns)
            | set(cdt_t.columns)
            | set(cc_t.columns)
            | set(any_t.columns)
        )
        - {"(sans variant)"}
    )

    print(f"=== {slug} ===")
    for variant in variants:
        visites = int(visites_t.at[slug, variant]) if variant in visites_t.columns else 0
        cdt = int(cdt_t.at[slug, variant]) if variant in cdt_t.columns else 0
        cc = int(cc_t.at[slug, variant]) if variant in cc_t.columns else 0
        any_ = int(any_t.at[slug, variant]) if variant in any_t.columns else 0

        taux_cdt = cdt / visites * 100 if visites else 0
        taux_cc = cc / visites * 100 if visites else 0
        taux_global = any_ / visites * 100 if visites else 0

        print(f"  --- {variant} ---")
        print(f"    Visites uniques      : {visites}")
        print(f"    Consult. CDT         : {cdt} ({taux_cdt:.1f} %)")
        print(f"    Consult. CC          : {cc} ({taux_cc:.1f} %)")
        print(f"    Consult. CDT ou CC   : {any_} (taux global : {taux_global:.1f} %)")
    print()


# Tri des slugs par nombre total de visites (tous variants confondus), décroissant
slugs_ordered = (
    visites_t.drop(index="TOTAL")
    .sum(axis=1)
    .sort_values(ascending=False)
    .index
)

for slug in slugs_ordered:
    print_detail_slug(slug, visites_t, cdt_t, cc_t, any_t)